# 4.1 计算图优化

通过编译器优化计算图，减少不必要的计算和内存访问，最大化硬件利用率。核心优化包括算子融合、死代码消除、常量折叠和内存布局优化。

## 什么是计算图优化？

计算图优化是对深度学习模型的数据流图进行等价变换，在保证数学语义不变的前提下，减少冗余计算和内存访问的编译器技术。具体包括：
- **算子融合（Operator Fusion）**：将多个连续算子合并为单个算子执行，消除中间张量的全局内存读写
- **常量折叠（Constant Folding）**：在编译期预计算仅依赖常量的子图
- **死代码消除（DCE）**：移除对最终输出无贡献的计算分支
- **内存布局优化**：调整张量排布方式以匹配硬件访存偏好

## 为什么用计算图优化？

1. **减少内核启动开销**：GPU上每次kernel launch需约$5\!-\!10\mu s$，融合后可从$N$次降至$O(1)$次
2. **减少内存带宽消耗**：未融合时中间张量需写回全局内存，融合后中间结果留在寄存器中
3. **增加计算密度**：融合后算术强度$I = \frac{\text{FLOPs}}{\text{Bytes}}$更大，更接近硬件峰值性能

## 计算图优化的数学原理

**算子融合**：$y = g(f(x)) \Leftrightarrow y = h(x)$，其中$h = g \circ f$，关键收益在于中间结果$f(x)$不再写入全局内存：
$$\text{Memory}_{\text{fused}} = |x| + |y| \ll |x| + |f(x)| + |y| = \text{Memory}_{\text{unfused}}$$

**Roofline模型**：算术强度$I$决定性能上界：
$$P_{\max} = \begin{cases} I \cdot \text{BW} & I < I_{\text{threshold}} \text{ (memory-bound)} \\ \text{Peak FLOPS} & I \geq I_{\text{threshold}} \text{ (compute-bound)} \end{cases}$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 算子融合（Operator Fusion）

#### 什么是算子融合？

将多个连续算子合并为单个算子执行，减少中间结果的内存读写。这是编译器优化中最重要的一项。

#### 为什么算子融合有效？

1. **消除中间张量I/O**：未融合时中间结果需写回HBM再读回，融合后留在寄存器/共享内存中
2. **减少内核启动开销**：GPU每次kernel launch需约$5-10\mu s$，融合后从$N$次降至$O(1)$次
3. **提高算术强度**：融合后$I = \frac{\text{FLOPs}}{\text{Bytes}}$更大，更容易接近硬件峰值

#### 算子融合的数学原理

对于顺序执行的算子链 $y = op_k \circ \cdots \circ op_1(x)$，融合后等价于：
$$y = h(x), \quad h = op_k \circ \cdots \circ op_1$$

内存访问量对比：
$$\text{Memory}_{\text{fused}} = |x| + |y| \ll |x| + \sum_{i=1}^{k-1}|\text{intermediate}_i| + |y| = \text{Memory}_{\text{unfused}}$$

常见融合模式：
- **Linear + ReLU**：$y = \max(0, Wx+b)$，消除中间激活的HBM写入
- **MatMul + Bias + GELU**：$y = \text{GELU}(Wx+b)$，三合一融合
- **Softmax + Dropout**：$y = \text{Dropout}(\text{Softmax}(z))$，概率值不落HBM

In [ ]:
class UnfusedLinearReLU(nn.Module):
    """未融合: Linear + ReLU 两个独立算子"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        h = self.linear(x)
        return self.relu(h)

class FusedLinearReLU(nn.Module):
    """融合: Linear + ReLU 合并为单个kernel"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        self.bias = nn.Parameter(torch.randn(out_dim))

    def forward(self, x):
        return F.linear(x, self.weight, self.bias).clamp(min=0)

in_dim, out_dim = 512, 256
unfused = UnfusedLinearReLU(in_dim, out_dim)
fused = FusedLinearReLU(in_dim, out_dim)
fused.weight.data.copy_(unfused.linear.weight.data)
fused.bias.data.copy_(unfused.linear.bias.data)

x = torch.randn(128, in_dim)

with torch.no_grad():
    out_unfused = unfused(x)
    out_fused = fused(x)
    diff = (out_unfused - out_fused).norm() / out_unfused.norm() * 100

n_iters = 1000
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = unfused(x)
    unfused_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = fused(x)
    fused_time = time.time() - start

print(f"=== 算子融合: Linear + ReLU ===")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"未融合时间: {unfused_time:.4f}s")
print(f"融合时间: {fused_time:.4f}s")
print(f"加速比: {unfused_time/fused_time:.2f}x")
print(f"\n融合节省: 1次中间张量的内存写入+读取")

class UnfusedLinearBNReLU(nn.Module):
    """未融合: Linear + BatchNorm + ReLU"""
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.linear(x)))

class FusedLinearBNReLU(nn.Module):
    """融合: BN参数吸收到Linear中 + ReLU"""
    def __init__(self, linear, bn):
        super().__init__()
        with torch.no_grad():
            bn_var = bn.running_var
            bn_mean = bn.running_mean
            bn_weight = bn.weight
            bn_bias = bn.bias
            eps = bn.eps
            scale = bn_weight / torch.sqrt(bn_var + eps)
            self.weight = nn.Parameter(linear.weight * scale.unsqueeze(1))
            self.bias = nn.Parameter(linear.bias * scale + bn_bias - bn_mean * scale)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias).clamp(min=0)

dim = 256
unfused_lbr = UnfusedLinearBNReLU(dim)
unfused_lbr.eval()
fused_lbr = FusedLinearBNReLU(unfused_lbr.linear, unfused_lbr.bn)

x = torch.randn(64, dim)
with torch.no_grad():
    out_u = unfused_lbr(x)
    out_f = fused_lbr(x)
    diff = (out_u - out_f).norm() / out_u.norm() * 100

print(f"\n=== 算子融合: Linear + BatchNorm + ReLU ===")
print(f"输出差异: {diff:.6f}%")
print(f"BN参数吸收到Linear后: 3个算子 -> 1个算子")
print(f"节省: 2次中间张量读写 + BN计算")

### 常量折叠 & 死代码消除

#### 什么是常量折叠？

如果计算图中的某个子节点的所有输入都是编译期已知的常量（如模型参数、固定标量），则可以在编译阶段预先计算该子节点，运行时直接使用结果。

#### 什么是死代码消除？

如果计算图中某个节点的输出对最终结果没有任何贡献（如被乘以0、未被下游节点引用），则该节点及其依赖的子图可被安全移除。

#### 为什么它们有效？

1. **常量折叠**：将运行时计算$O(	ext{FLOPs})$转化为编译期预计算$O(1)$，同时减少运行时参数量
2. **死代码消除**：直接移除无效计算，减少不必要的FLOPs和内存分配
3. **组合优化**：常量折叠可能暴露新的死代码，死代码消除也可能暴露新的常量折叠机会，两者交替进行直至收敛

#### 数学原理

- 常量折叠：$c = f(a, b)$，若$a, b$为常量$\Rightarrow$编译期计算$c = f(a, b)$，运行时直接查表
- 死代码消除：若节点$v$满足$\frac{\partial \mathcal{L}}{\partial v} = 0$（对损失无梯度贡献），或$v$乘以零因子（$v 	imes 0 = 0$），则$v$可安全移除

In [ ]:
class ModelWithRedundancy(nn.Module):
    """包含冗余计算的模型"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.fc3 = nn.Linear(dim, dim)
        self.constant_scale = nn.Parameter(torch.tensor(2.0), requires_grad=False)
        self.constant_bias = nn.Parameter(torch.tensor(0.5), requires_grad=False)

    def forward(self, x):
        h = self.fc1(x)
        dead_branch = self.fc2(x) * 0
        h = h * self.constant_scale + self.constant_bias
        h = F.relu(h)
        h = self.fc3(h)
        return h + dead_branch

class OptimizedModel(nn.Module):
    """优化后: 常量折叠 + 死代码消除"""
    def __init__(self, original):
        super().__init__()
        self.fc1 = nn.Linear(original.fc1.in_features, original.fc1.out_features)
        self.fc3 = nn.Linear(original.fc3.in_features, original.fc3.out_features)
        with torch.no_grad():
            self.fc1.weight.copy_(original.fc1.weight * original.constant_scale)
            self.fc1.bias.copy_(original.fc1.bias * original.constant_scale + original.constant_bias)
            self.fc3.weight.copy_(original.fc3.weight)
            self.fc3.bias.copy_(original.fc3.bias)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc3(h)

dim = 256
original = ModelWithRedundancy(dim)
optimized = OptimizedModel(original)

x = torch.randn(32, dim)
with torch.no_grad():
    out_orig = original(x)
    out_opt = optimized(x)
    diff = (out_orig - out_opt).norm() / out_orig.norm() * 100

orig_params = sum(p.numel() for p in original.parameters())
opt_params = sum(p.numel() for p in optimized.parameters())

n_iters = 500
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = original(x)
    orig_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = optimized(x)
    opt_time = time.time() - start

print(f"=== 常量折叠 + 死代码消除 ===")
print(f"输出差异: {diff:.6f}%")
print(f"原始参数: {orig_params:,}")
print(f"优化后参数: {opt_params:,} (减少{(1-opt_params/orig_params)*100:.0f}%)")
print(f"原始时间: {orig_time:.4f}s")
print(f"优化后时间: {opt_time:.4f}s (加速{orig_time/opt_time:.2f}x)")
print(f"\n优化内容:")
print(f"1. 常量折叠: scale*bias预计算合并到fc1权重")
print(f"2. 死代码消除: fc2(x)*0 永远为0，移除")
print(f"3. 算子融合: fc1+scale+bias+relu 合并")

### PyTorch torch.compile 实践

#### 什么是torch.compile？

PyTorch 2.0引入的即时编译（JIT）框架，由三个核心组件构成：
- **TorchDynamo**：在Python字节码层面捕获计算图，支持动态控制流
- **TorchInductor**：后端代码生成器，将FX Graph编译为Triton（GPU）或C++（CPU）高效代码
- **AOTAutograd**：自动微分图构建，在前向图基础上自动生成反向图

#### 为什么用torch.compile？

1. **自动算子融合**：Inductor后端自动识别可融合的算子模式，生成融合kernel
2. **消除Python解释器开销**：编译后代码直接以C++/Triton执行，跳过Python调度
3. **硬件感知优化**：针对不同GPU架构生成最优kernel

#### 核心原理

torch.compile的优化流水线：
$$\text{Python Bytecode} \xrightarrow{\text{Dynamo}} \text{FX Graph} \xrightarrow{\text{AOTAutograd}} \text{Fwd+Bwd Graphs} \xrightarrow{\text{Inductor}} \text{Triton/C++ Code}$$

Inductor在FX Graph上执行算子融合、内存布局优化、循环分块等变换，最终生成硬件原生代码。

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, dim=128, n_heads=4, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }) for _ in range(n_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            h = layer['ln1'](x)
            h, _ = layer['attn'](h, h, h)
            x = x + h
            x = x + layer['ffn'](layer['ln2'](x))
        return x

model = SimpleTransformer(dim=128, n_heads=4, n_layers=4)
model.eval()
x = torch.randn(4, 32, 128)

with torch.no_grad():
    out_eager = model(x)

try:
    compiled_model = torch.compile(model, mode='reduce-overhead')
    with torch.no_grad():
        out_compiled = compiled_model(x)
    diff = (out_eager - out_compiled).norm() / out_eager.norm() * 100
    print(f"torch.compile编译成功")
    print(f"输出差异: {diff:.6f}%")

    n_iters = 100
    with torch.no_grad():
        start = time.time()
        for _ in range(n_iters):
            _ = model(x)
        eager_time = time.time() - start

        for _ in range(5):
            _ = compiled_model(x)
        start = time.time()
        for _ in range(n_iters):
            _ = compiled_model(x)
        compiled_time = time.time() - start

    print(f"Eager模式: {eager_time:.4f}s")
    print(f"Compiled模式: {compiled_time:.4f}s")
    print(f"加速比: {eager_time/compiled_time:.2f}x")
except Exception as e:
    print(f"torch.compile不可用: {e}")
    print(f"torch.compile需要PyTorch 2.0+")

print(f"\ntorch.compile优化内容:")
print(f"1. 自动算子融合")
print(f"2. 内存布局优化")
print(f"3. 减少Python开销")
print(f"4. 生成针对硬件优化的kernel")

### 产业级实战：torch.compile 多模式性能对比

PyTorch 2.0+ 的 `torch.compile` 是产业界最常用的图编译优化工具。它支持三种模式：
- `default`：平衡编译时间和运行时性能
- `reduce-overhead`：减少CUDA kernel launch开销（使用CUDAGraphs）
- `max-autotune`：尝试所有可能的kernel配置，选择最快的（编译最慢但运行最快）

In [ ]:
import time

class TransformerBlock(nn.Module):
    def __init__(self, dim=256, n_heads=4, ffn_dim=1024):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, dim),
        )
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

class SmallTransformer(nn.Module):
    def __init__(self, vocab_size=1000, dim=256, depth=4, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.Sequential(*[TransformerBlock(dim, n_heads) for _ in range(depth)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)

model = SmallTransformer(vocab_size=1000, dim=256, depth=4, n_heads=4)
model.eval()
dummy = torch.randint(0, 1000, (4, 64))

def benchmark_fn(fn, *args, warmup=5, runs=20):
    for _ in range(warmup):
        fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / runs * 1000
    return elapsed

eager_time = benchmark_fn(model, dummy)
print(f"Eager模式: {eager_time:.2f} ms")

modes = ['default', 'reduce-overhead', 'max-autotune']
results = {'eager': eager_time}
for mode in modes:
    try:
        compiled = torch.compile(model, mode=mode)
        with torch.no_grad():
            compiled_time = benchmark_fn(compiled, dummy)
        speedup = eager_time / compiled_time
        results[mode] = compiled_time
        print(f"torch.compile({mode}): {compiled_time:.2f} ms ({speedup:.2f}x)")
        del compiled
    except Exception as e:
        print(f"torch.compile({mode}): 失败 - {e}")

print(f"\n=== torch.compile 性能对比 ===")
print(f"{'模式':<20} {'延迟(ms)':<12} {'加速比':<10}")
print("-" * 42)
for name, t in results.items():
    speedup = f"{eager_time/t:.2f}x" if name != 'eager' else '1.00x'
    print(f"{name:<20} {t:<12.2f} {speedup:<10}")

print(f"\n产业界torch.compile使用建议:")
print(f"1. 开发阶段: eager模式 (调试方便)")
print(f"2. 部署阶段: reduce-overhead (GPU推理首选)")
print(f"3. 极致性能: max-autotune (编译慢但运行最快)")
print(f"4. 常见问题: 动态shape → 使用dynamic=True")
print(f"5. 调试: TORCH_LOGS='+dynamo' 查看编译日志")

### Graph Break 与编译陷阱

#### 什么是Graph Break？

当TorchDynamo遇到无法编译的Python代码时，会中断计算图捕获，回退到eager模式执行，这称为Graph Break。Graph Break会显著降低编译优化的效果。

#### 常见触发Graph Break的操作

1. **数据依赖的控制流**：`if x.sum() > 0:`（条件依赖张量值）
2. **动态shape操作**：`x.view(-1, x.size(1))`（shape依赖数据）
3. **Python副作用**：`print(x.shape)`、`assert x.dim() == 3`
4. **不支持的Python特性**：`eval()`、`exec()`、反射操作

#### 如何检测和解决Graph Break

```python
# 方法1: 使用TORCH_LOGS环境变量
import os
os.environ['TORCH_LOGS'] = '+dynamo'

# 方法2: 使用torch._dynamo.explain
explanation = torch._dynamo.explain(model)(x)
print(f'Graph breaks: {explanation.graph_break_count}')

# 方法3: 使用torch.compile的fullgraph=True强制无break
compiled = torch.compile(model, fullgraph=True)  # 有break则报错
```

#### torch.compile vs torch.export

| 特性 | torch.compile | torch.export |
|------|--------------|-------------|
| 目标 | 即时加速 | 导出可移植IR |
| 动态shape | 支持(dynamic=True) | 需显式声明dim |
| Graph Break | 自动回退eager | 不允许，必须修复 |
| 输出 | 运行时优化模型 | ExportedProgram |
| 部署场景 | 训练/推理加速 | 跨框架部署(C++/ONNX) |
| 典型用途 | GPU推理优化 | 端侧模型导出 |

In [ ]:
class ModelWithGraphBreak(nn.Module):
    """包含Graph Break的模型"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, x):
        h = self.fc1(x)
        if h.sum() > 0:
            h = self.fc2(h)
        else:
            h = h * 2
        return h

class ModelFixed(nn.Module):
    """修复Graph Break: 用torch.where替代数据依赖控制流"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, x):
        h = self.fc1(x)
        condition = (h.sum() > 0)
        h_fc2 = self.fc2(h)
        h = torch.where(condition, h_fc2, h * 2)
        return h

dim = 64
model_break = ModelWithGraphBreak(dim)
model_fixed = ModelFixed(dim)
model_break.eval()
model_fixed.eval()

x = torch.randn(4, dim)

with torch.no_grad():
    out_break = model_break(x)
    out_fixed = model_fixed(x)

print(f"=== Graph Break 检测与修复 ===")
print(f"原始模型输出: {out_break[0, :5]}")
print(f"修复模型输出: {out_fixed[0, :5]}")

try:
    compiled_break = torch.compile(model_break, fullgraph=True)
    with torch.no_grad():
        compiled_break(x)
    print(f"\nfullgraph=True编译: 成功 (无Graph Break)")
except Exception as e:
    print(f"\nfullgraph=True编译: 失败 (存在Graph Break)")
    print(f"  原因: 数据依赖的控制流 if h.sum() > 0")

try:
    compiled_fixed = torch.compile(model_fixed, fullgraph=True)
    with torch.no_grad():
        compiled_fixed(x)
    print(f"修复后fullgraph=True编译: 成功")
except Exception as e:
    print(f"修复后fullgraph=True编译: 失败 - {e}")

print(f"\nGraph Break修复策略:")
print(f"1. 数据依赖控制流 → torch.where / torch.cond")
print(f"2. 动态shape → 固定shape或使用dynamic=True")
print(f"3. Python副作用 → 移除print/assert或用torch._dynamo.disable")
print(f"4. 不支持操作 → 用@torch.compiler.disable标记子图")

### 公共子表达式消除（CSE）与代数简化

#### 什么是CSE？

如果计算图中存在多个相同的子表达式（相同输入+相同操作），只需计算一次并复用结果。

#### 什么是代数简化？

利用代数恒等式简化计算：
- $x + 0 = x$（加零消除）
- $x \times 1 = x$（乘一消除）
- $x \times 0 = 0$（乘零消除）
- $\text{concat}(x, x) = \text{repeat}(x, 2)$（合并重复）
- $\text{reshape}(\text{reshape}(x, s_1), s_2) = \text{reshape}(x, s_2)$（消除冗余reshape）

#### 为什么它们有效？

1. **CSE**：避免重复计算，减少FLOPs。在大模型中，同一子表达式可能被多处引用
2. **代数简化**：零代价消除无效计算，特别是残差连接中常见的$+0$和$\times 1$模式
3. **组合效果**：CSE+代数简化+常量折叠+DCE交替执行，可消除大量冗余

In [ ]:
class ModelWithCSE(nn.Module):
    """包含公共子表达式的模型"""
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        a = self.fc(x)
        b = self.fc(x)
        return a + b

class ModelCSEOptimized(nn.Module):
    """CSE优化后: 相同计算只执行一次"""
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        a = self.fc(x)
        return a * 2

class ModelWithAlgebraicSimp(nn.Module):
    """包含代数冗余的模型"""
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        h = self.fc(x)
        h = h + 0
        h = h * 1
        h = h.reshape(x.shape)
        return h

class ModelAlgebraicOptimized(nn.Module):
    """代数简化后: 消除+0, *1, 冗余reshape"""
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        return self.fc(x)

dim = 256
x = torch.randn(32, dim)

models = {
    'CSE原始': (ModelWithCSE(dim), ModelCSEOptimized(dim)),
    '代数简化原始': (ModelWithAlgebraicSimp(dim), ModelAlgebraicOptimized(dim)),
}

print(f"=== CSE与代数简化 ===")
for name, (orig, opt) in models.items():
    orig.eval()
    opt.eval()
    with torch.no_grad():
        out_o = orig(x)
        out_p = opt(x)
        diff = (out_o - out_p).norm() / out_o.norm() * 100

        start = time.time()
        for _ in range(500):
            orig(x)
        orig_t = time.time() - start

        start = time.time()
        for _ in range(500):
            opt(x)
        opt_t = time.time() - start

    print(f"{name}: 差异={diff:.6f}%, 加速={orig_t/opt_t:.2f}x")

print(f"\n编译器优化流水线（迭代直至收敛）:")
print(f"  常量折叠 → CSE → 代数简化 → DCE → 常量折叠 → ...")
print(f"  每一轮可能暴露新的优化机会，需迭代执行")